# Generic GLUE Quantized BERT Runner

Run either MRPC or CoLA from this notebook by changing `TASK_NAME`.


In [1]:
!pip install transformers==4.35.2
!pip install datasets evaluate fsspec


In [2]:
import transformers
print(transformers.__version__)


4.35.2


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import sys
from pathlib import Path

os.environ["HF_DATASETS_OFFLINE"] = "0"

PROJECT_DIR_PATH = "/content/drive/MyDrive/mrcp-tr-ptq"
PROJECT_DIR = Path(PROJECT_DIR_PATH)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}
!ls {PROJECT_DIR_PATH}


/content/drive/MyDrive/mrcp-tr-ptq
copy_of_bert_glue_mrcp.ipynb  output		      quant_config.json
copy_of_bert_glue_mrcp.py     __pycache__	      README.md
mrcp_quant		      quant_config_cola.json  run_glue_quant.py


## Task And Config

Use `TASK_NAME = "mrpc"` or `TASK_NAME = "cola"`. The matching default config file is selected below.


In [5]:
TASK_NAME = "cola"  # "mrpc" or "cola"

CONFIG_BY_TASK = {
    "mrpc": "quant_config.json",
    "cola": "quant_config_cola.json",
}

CONFIG_PATH = PROJECT_DIR / CONFIG_BY_TASK[TASK_NAME]
print("TASK_NAME:", TASK_NAME)
print("CONFIG_PATH:", CONFIG_PATH)


TASK_NAME: cola
CONFIG_PATH: /content/drive/MyDrive/mrcp-tr-ptq/quant_config_cola.json


## Imports


In [6]:
import torch
from transformers import AutoModelForSequenceClassification, BertConfig, BertTokenizerFast

from mrcp_quant import (
    apply_experiment_config,
    apply_layer_quant_overrides,
    get_task_spec,
    load_experiment_config,
    resolve_q_module_list,
    save_experiment_result,
)
from run_glue_quant import calibrate_model, evaluate_model, optimize_scale_factors


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


## Model Initialization


In [7]:
experiment_config = load_experiment_config(CONFIG_PATH)
experiment_config["task_name"] = TASK_NAME
apply_experiment_config(experiment_config)

task = get_task_spec(experiment_config.get("task_name", "mrpc"))
model_name = experiment_config.get("model_name", task.default_model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("task", task.name)
print("model_name", model_name)
print("device", device)

tokenizer = BertTokenizerFast.from_pretrained(model_name)
hf_config = BertConfig.from_pretrained(model_name)
hf_model = AutoModelForSequenceClassification.from_pretrained(model_name)

model = task.model_class(hf_config)
applied_layer_quant_overrides = apply_layer_quant_overrides(model, experiment_config)
if applied_layer_quant_overrides:
    print("Applied layer quantization overrides:", applied_layer_quant_overrides)

res = model.load_state_dict(hf_model.state_dict(), strict=False)
print("Missing keys:", len(res.missing_keys))
print("Unexpected keys:", len(res.unexpected_keys))
print("Missing examples:", res.missing_keys[:30])

model.to(device)


task cola
model_name geckos/bert-base-uncased-finetuned-glue-cola
device cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Missing keys: 1
Unexpected keys: 0
Missing examples: ['bert.embeddings.position_ids']


CustomBertForSequenceClassification(
  (bert): CustomBertModel(
    (embeddings): CustomBertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): QLayerNorm(
        (768,), eps=1e-12, elementwise_affine=True
        (in_obs_normalize): MinMaxObserver()
        (in_obs): MinMaxObserver()
        (w_obs): MinMaxObserver()
        (b_obs): MinMaxObserver()
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CustomBertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CustomBertLayer(
          (attention): CustomBertAttention(
            (self): CustomBertSelfAttention(
              (query): QuantizedLinear(
                in_features=768, out_features=768, bias=True
                (in_obs): MinMaxObserver()
                (w_obs): MinMaxObserver()
                (b_obs): MinMaxObserver()
              )
             

## Quantization Setup


In [8]:
q_module_list = resolve_q_module_list(
    experiment_config.get("q_module_list", ["QLayerNorm"])
)

model.set_q_module_list(q_module_list)
model.set_quant()

note = []
for name, module in model.named_modules():
    q = getattr(module, "quant", None)
    opt = getattr(module, "is_opt_scale", None)
    if (q is True) or (opt is True):
        note.append((name, type(module).__name__, q, opt))

print("modules not in pure-float mode:", len(note))
print(*note[:50], sep="\n")


modules not in pure-float mode: 25
('bert.embeddings.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.0.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.0.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.1.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.1.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.2.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.2.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.3.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.3.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.4.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.4.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.5.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.5.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer

## Calibration And Scale Optimization


In [9]:
calibrate_model(model, task, tokenizer, q_module_list, experiment_config, device)
optimize_scale_factors(model, task, tokenizer, q_module_list, experiment_config, device)


README.md: 0.00B [00:00, ?B/s]

Calibrating quantization parameters...
0|

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:907: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29|30|31|
Quantization parameters were set
Optimizing model scale factors...
0|____________rmse:0.1314782053232193_____________
_______________________________________________
scale_factor:0.5, nrmse:11.578473091125488
scale_factor:0.5306122448979592, nrmse:9.55897045135498
scale_factor:0.5612244897959183, nrmse:7.926259994506836
scale_factor:0.5918367346938775, nrmse:6.597748279571533
scale_factor:0.6224489795918368, nrmse:5.505231857299805
scale_factor:0.653061224489796, nrmse:4.596648216247559
scale_factor:0.6836734693877551, nrmse:3.8316235542297363
scale_factor:0.7142857142857143, nrmse:3.1820077896118164
scale_factor:0.7448979591836735, nrmse:2.625152587890625
scale_factor:0.7755102040816326, nrmse:2.1440229415893555
scale_factor:0.8061224489795918, nrmse:1.7253117561340332
scale_factor:0.8367346938775511, nrmse:1.3593655824661255
scale_factor:0.8673469387755102, nrmse:1.0390347242355347
scale_factor:0.897

## Evaluation


In [10]:
metrics, average_loss, num_examples = evaluate_model(
    model,
    task,
    tokenizer,
    experiment_config,
    device,
)

primary_metric_value = metrics[task.primary_metric]
print("metrics", metrics)
print(f"Final {task.primary_metric}:", primary_metric_value)
print("Final Loss:", average_loss)


 15%|█▌        | 10/65 [00:01<00:06,  8.50it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:907: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 18%|█▊        | 12/65 [00:01<00:05,  9.14it/s]


[Batch 10] Interim matthews_correlation: 0.4996



 29%|██▉       | 19/65 [00:02<00:04, 10.07it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:907: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 32%|███▏      | 21/65 [00:02<00:04, 10.07it/s]


[Batch 20] Interim matthews_correlation: 0.5143



 45%|████▍     | 29/65 [00:03<00:03, 10.44it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:907: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 48%|████▊     | 31/65 [00:03<00:03, 10.34it/s]

...


[Batch 50] Interim matthews_correlation: 0.5251



 91%|█████████ | 59/65 [00:06<00:00, 10.45it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:907: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
 94%|█████████▍| 61/65 [00:06<00:00, 10.31it/s]


[Batch 60] Interim matthews_correlation: 0.5304



66it [00:06,  9.65it/s]                        

metrics {'matthews_correlation': np.float64(0.5265703302665122)}
Final matthews_correlation: 0.5265703302665122
Final Loss: 0.5571589336702318


## Save Result


In [11]:
result_path = save_experiment_result(
    accuracy=primary_metric_value,
    loss=average_loss,
    configuration=experiment_config,
    quantized=experiment_config.get("q_module_list", []),
    output_dir=PROJECT_DIR / "output",
    extra={
        "task_name": task.name,
        "model_name": model_name,
        "metrics": metrics,
        "primary_metric_name": task.primary_metric,
        "primary_metric_value": primary_metric_value,
        "num_val_examples": num_examples,
    },
)
print("Saved results:", result_path)


Saved results: /content/drive/MyDrive/mrcp-tr-ptq/output/result_20260501_103942.json
